# Preprocesamiento y transformación de los datos

## Objetivo

Este notebook desarrolla la etapa de **Preprocesamiento y Transformación** dentro de la metodología KDD.

Su finalidad consiste en depurar el conjunto integrado, eliminar variables que no aportan información útil, examinar el significado de los valores ausentes, verificar la presencia de duplicados y generar el **conjunto procesado `viviendas_limpio.csv`**, que emplearán el análisis exploratorio y la modelización.

Una decisión central de esta fase es **no aplicar una imputación general de los valores ausentes**. Cuando una variable relevante presenta datos no informados, estos se mantienen en el conjunto procesado y, si fuera preciso, su tratamiento se realizará más adelante de manera específica para cada análisis.


## 1. Importación de librerías

Se utilizan `Path` para gestionar las rutas y `pandas`/`numpy` para la manipulación y revisión del conjunto de datos.

In [ ]:
# =====================================================
# IMPORTACIÓN DE LIBRERÍAS
# =====================================================

from pathlib import Path

import numpy as np
import pandas as pd

### Interpretación

Las librerías necesarias para el preprocesamiento se han cargado de forma correcta. `numpy` y `pandas` permiten trabajar con valores ausentes, tipos de datos y transformaciones, mientras que `Path` asegura una gestión coherente de las rutas del proyecto.


## 2. Configuración y carga de los datos

Se conserva la misma estructura de carpetas del Notebook 01 y se vuelven a cargar los archivos originales. Así, el notebook puede ejecutarse de forma independiente sin depender de variables almacenadas en memoria por una ejecución previa.


In [ ]:
# =====================================================
# CONFIGURACIÓN DE RUTAS Y CARGA
# =====================================================

# =====================================================
# DETECCIÓN DE LA RAÍZ DEL PROYECTO
# =====================================================

# Estructura esperada:
# TFM-Analisis-Inmobiliario/
# ├── data/raw
# ├── data/processed
# ├── notebooks
# └── reports/{figures,tables}
#
# La detección funciona tanto si VS Code ejecuta el notebook
# desde la raíz del proyecto como desde la carpeta notebooks/.
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        ruta for ruta in [cwd, *cwd.parents]
        if (ruta / "data" / "raw").is_dir()
        and (ruta / "notebooks").is_dir()
        and (ruta / "reports").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se ha podido localizar la raíz del proyecto. "
        "Se esperaba encontrar las carpetas data/raw, notebooks y reports "
        "en el directorio actual o en alguno de sus directorios superiores."
    )

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

excel_files = sorted(
    p for p in RAW_DATA_DIR.glob("* 2026.xlsx")
    if "INE" not in p.parts
)

if not excel_files:
    raise FileNotFoundError(
        "No se han encontrado archivos de ciudades. "
        f"Revisa que los Excel estén dentro de: {RAW_DATA_DIR}"
    )

if len(excel_files) != 10:
    print(
        f"AVISO: se han encontrado {len(excel_files)} archivos; "
        "se esperaban 10 archivos de ciudades."
    )

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Datos originales: {RAW_DATA_DIR}")
print(f"Datos procesados: {PROCESSED_DATA_DIR}")

df_viviendas = None

for file in excel_files:
    df_ciudad = pd.read_excel(file)
    df_ciudad["ciudad"] = file.stem.replace(" 2026", "")

    if df_viviendas is None:
        df_viviendas = df_ciudad
    else:
        df_viviendas = pd.concat(
            [df_viviendas, df_ciudad],
            ignore_index=True,
            sort=False,
        )

    del df_ciudad

print(f"Archivos cargados: {len(excel_files)}")
print(f"Registros: {df_viviendas.shape[0]:,}")
print(f"Variables: {df_viviendas.shape[1]}")


### Interpretación

La integración reproduce el punto de partida de la fase anterior: **55.653 registros y 535 variables**. Esta comprobación garantiza que el preprocesamiento se aplica sobre el conjunto completo y no sobre una muestra parcial.


## 3. Diagnóstico inicial de calidad

### 3.1 Tipos de datos

Se examina la distribución de tipos de datos para conocer la naturaleza general de las variables y detectar posibles formatos que requieran atención posterior.


In [ ]:
# =====================================================
# TIPOS DE DATOS
# =====================================================

tipos_datos = (
    df_viviendas.dtypes.astype(str)
    .value_counts()
    .rename_axis("Tipo de dato")
    .reset_index(name="Número de variables")
)

tipos_datos

### Interpretación

El conjunto incluye variables numéricas, booleanas, categóricas y textuales. Esta diversidad es coherente con la información de los anuncios inmobiliarios y obliga a tratar cada grupo según su significado, en vez de aplicar una única transformación a todas las columnas.


### 3.2 Valores ausentes

Se calcula el número y el porcentaje de valores nulos en cada variable. La finalidad es distinguir entre columnas completamente vacías, variables muy poco informadas y variables relevantes con ausencias moderadas.


In [ ]:
# =====================================================
# ANÁLISIS DE VALORES NULOS
# =====================================================

valores_nulos = pd.DataFrame({
    "Valores nulos": df_viviendas.isna().sum(),
    "Porcentaje (%)": (df_viviendas.isna().mean() * 100).round(2),
})

valores_nulos = (
    valores_nulos[valores_nulos["Valores nulos"] > 0]
    .sort_values("Porcentaje (%)", ascending=False)
)

intervalos = [0, 1, 25, 50, 75, 90, 100]
etiquetas = [
    "< 1 %",
    "1 % - < 25 %",
    "25 % - < 50 %",
    "50 % - < 75 %",
    "75 % - < 90 %",
    "90 % - < 100 %",
]

clasificacion_nulos = valores_nulos.copy()
clasificacion_nulos["Intervalo"] = pd.cut(
    clasificacion_nulos["Porcentaje (%)"],
    bins=intervalos,
    labels=etiquetas,
    right=False,
)
clasificacion_nulos["Intervalo"] = (
    clasificacion_nulos["Intervalo"]
    .cat.add_categories("100 %")
)
clasificacion_nulos.loc[
    clasificacion_nulos["Porcentaje (%)"] == 100,
    "Intervalo"
] = "100 %"

resumen_nulos = (
    clasificacion_nulos["Intervalo"]
    .value_counts()
    .reset_index()
)
resumen_nulos.columns = ["Intervalo de valores nulos", "Número de variables"]

display(valores_nulos.head(20))
display(resumen_nulos)

### Interpretación

La falta de información se concentra sobre todo en variables auxiliares y multimedia. Sobresalen **98 variables completamente vacías** y un número elevado de columnas con más del 90 % de valores nulos. Este hecho justifica realizar primero una depuración estructural antes de decidir el tratamiento de los nulos en las variables relevantes.


## 4. Depuración estructural de variables

### 4.1 Eliminación de variables completamente vacías

Una variable con el 100 % de valores nulos no aporta información al análisis. Antes de eliminarla, se revisa el grupo al que pertenece.


In [ ]:
# =====================================================
# VARIABLES COMPLETAMENTE VACÍAS
# =====================================================

variables_vacias = valores_nulos[
    valores_nulos["Porcentaje (%)"] == 100
]

categorias_vacias = (
    variables_vacias.index
    .str.split("/")
    .str[0]
    .value_counts()
    .rename_axis("Categoría")
    .reset_index(name="Número de variables")
)

display(categorias_vacias)

df_viviendas = df_viviendas.drop(columns=variables_vacias.index)

print(f"Variables eliminadas: {len(variables_vacias)}")
print(f"Variables restantes: {df_viviendas.shape[1]}")

### Interpretación

La práctica totalidad de las columnas completamente vacías pertenece a información multimedia o a etiquetas auxiliares. Su eliminación no implica perder características útiles de las viviendas y reduce la dimensionalidad del conjunto.


### 4.2 Revisión de variables con más del 90 % de valores nulos

Tras retirar las columnas vacías, se identifican las variables extremadamente incompletas. Las variables multimedia, etiquetas y distintivos se tratan por separado; el resto se revisa por contenido antes de su eliminación.


In [ ]:
# =====================================================
# VARIABLES CON MÁS DEL 90 % DE NULOS
# =====================================================

valores_nulos = pd.DataFrame({
    "Valores nulos": df_viviendas.isna().sum(),
    "Porcentaje (%)": (df_viviendas.isna().mean() * 100).round(2),
})

valores_nulos = valores_nulos[
    valores_nulos["Valores nulos"] > 0
].sort_values("Porcentaje (%)", ascending=False)

variables_mas_90 = valores_nulos[
    valores_nulos["Porcentaje (%)"] >= 90
]

variables_revision = variables_mas_90[
    ~variables_mas_90.index.str.startswith(
        ("multimedia/", "labels/", "ribbons/")
    )
]

display(variables_revision)

for variable in variables_revision.index:
    print(f"\n{variable}")
    print(df_viviendas[variable].dropna().value_counts().head(10))

df_viviendas = df_viviendas.drop(columns=variables_revision.index)

print(f"Variables revisadas y eliminadas: {len(variables_revision)}")
print(f"Variables restantes: {df_viviendas.shape[1]}")

### Interpretación

Las variables no multimedia que superan el 90 % de valores ausentes muestran una disponibilidad muy reducida. Entre ellas figuran `contactInfo/phone1/internationalPrefix`, `highlightComment`, `parkingSpace/parkingSpacePrice` y `newDevelopmentFinished`.

Su información aparece en menos del 6 % de los anuncios y describe situaciones muy concretas o aspectos auxiliares del anuncio. Por ese motivo se eliminan en esta fase. La decisión no se apoya solo en el porcentaje de nulos, sino también en el contenido y en su utilidad para el objetivo comparativo del TFM.


### 4.3 Eliminación de variables multimedia

Las variables cuyo nombre comienza por `multimedia/` describen imágenes, vídeos y otros recursos de presentación del anuncio. No representan características inmobiliarias empleadas en los objetivos del TFM.


In [ ]:
# =====================================================
# ELIMINACIÓN DE VARIABLES MULTIMEDIA
# =====================================================

variables_multimedia = df_viviendas.columns[
    df_viviendas.columns.str.startswith("multimedia/")
]

df_viviendas = df_viviendas.drop(columns=variables_multimedia)

print(f"Variables multimedia eliminadas: {len(variables_multimedia)}")
print(f"Variables restantes: {df_viviendas.shape[1]}")

### Interpretación

La eliminación del bloque multimedia provoca la mayor reducción de dimensionalidad. Se conserva la información vinculada al inmueble y se retiran columnas técnicas de presentación que no resultan necesarias para estudiar el precio de oferta.


### 4.4 Información administrativa y del anuncio

Se revisan variables relacionadas con contacto, etiquetas, distintivos, referencias y miniaturas. Solo se conservan `contactInfo/userType` y `contactInfo/contactMethod`, por tratarse de variables categóricas interpretables sobre el tipo de anunciante y el canal de contacto.


In [ ]:
# =====================================================
# DEPURACIÓN DE INFORMACIÓN DEL ANUNCIO
# =====================================================

categorias_anuncio = (
    "contactInfo/",
    "labels/",
    "ribbons/",
    "externalReference",
    "thumbnail",
    "highlight",
)

variables_anuncio = [
    columna
    for columna in df_viviendas.columns
    if columna.startswith(categorias_anuncio)
]

variables_conservar = [
    "contactInfo/userType",
    "contactInfo/contactMethod",
]

variables_eliminar = [
    variable
    for variable in variables_anuncio
    if variable not in variables_conservar
]

df_viviendas = df_viviendas.drop(columns=variables_eliminar)

print(f"Variables del anuncio eliminadas: {len(variables_eliminar)}")
print(f"Variables conservadas de este bloque: {len(variables_conservar)}")
print(f"Variables totales tras la depuración: {df_viviendas.shape[1]}")

### Interpretación

Después de la depuración estructural, el conjunto queda reducido a **62 variables**. La reducción se fundamenta en el significado de las columnas y no solo en su porcentaje de nulos: se prioriza conservar información potencialmente útil sobre las viviendas, su localización, el precio y algunas características del anuncio.


## 5. Revisión de los valores ausentes en las variables conservadas

Una vez eliminadas las variables estructuralmente poco útiles, se analiza el significado de los nulos que permanecen. El objetivo no es imputarlos todavía, sino determinar si representan ausencia real de una característica, información no aplicable o información no proporcionada.


### 5.1 Reducciones de precio

Las variables de reducción de precio aparecen como un bloque: cuando una vivienda ha sufrido una bajada, se informa el conjunto de campos relacionados; cuando no la ha experimentado, esos campos quedan vacíos.


In [ ]:
# =====================================================
# REDUCCIONES DE PRECIO
# =====================================================

variables_price_drop = [
    "dropDate",
    "priceDropValue",
    "priceDropPercentage",
    "priceInfo/price/priceDropInfo/formerPrice",
    "priceInfo/price/priceDropInfo/priceDropValue",
    "priceInfo/price/priceDropInfo/priceDropPercentage",
]

resumen_price_drop = pd.DataFrame({
    "Valores no nulos": df_viviendas[variables_price_drop].count(),
    "Valores nulos": df_viviendas[variables_price_drop].isna().sum(),
    "Porcentaje de nulos (%)": (
        df_viviendas[variables_price_drop].isna().mean() * 100
    ).round(2),
})

display(resumen_price_drop)
display(df_viviendas[variables_price_drop].dropna(how="all").head(10))

### Interpretación

Los nulos de este bloque se interpretan como **no aplicable** cuando el anuncio no presenta una reducción de precio. Se mantienen como nulos para no inventar valores y conservar el significado original de los datos.


### 5.2 Aparcamiento

Las columnas de `parkingSpace` solo se informan cuando el anuncio incluye el bloque de aparcamiento. Se revisa de forma conjunta la disponibilidad y si la plaza está incluida en el precio.


In [ ]:
# =====================================================
# VARIABLES DE APARCAMIENTO
# =====================================================

variables_parking = [
    "parkingSpace/hasParkingSpace",
    "parkingSpace/isParkingSpaceIncludedInPrice",
]

resumen_parking = pd.DataFrame({
    "Valores no nulos": df_viviendas[variables_parking].count(),
    "Valores nulos": df_viviendas[variables_parking].isna().sum(),
    "Porcentaje de nulos (%)": (
        df_viviendas[variables_parking].isna().mean() * 100
    ).round(2),
})

display(resumen_parking)
display(df_viviendas[variables_parking].dropna(how="all").head(15))

### Interpretación

Las dos variables de aparcamiento siguen el mismo patrón de disponibilidad. En el dataset limpio se conservan sus valores originales. Más adelante, cuando se construya la variable de modelización `hasParking`, la ausencia del bloque se codificará de forma explícita y documentada sin modificar el CSV original.


### 5.3 Características y variables estructurales

Las variables del bloque `features/`, junto con `hasLift` y `exterior`, pueden contener `True`, `False` y `NaN`. En estos casos, `False` ya expresa de forma explícita la ausencia de la característica, mientras que `NaN` significa que la información no está disponible.

También se revisa `floor`, que presenta valores ausentes y diferentes formas de representar la planta. Su tratamiento se realizará posteriormente de manera específica en el notebook de modelización.


In [ ]:
# =====================================================
# FEATURES Y VARIABLES ESTRUCTURALES
# =====================================================

variables_features = [
    "features/hasGarden",
    "features/hasTerrace",
    "features/hasSwimmingPool",
    "features/hasBoxRoom",
    "features/hasAirConditioning",
]

variables_estructurales = [
    "floor",
    "hasLift",
    "exterior",
]

variables_revision_nulos = variables_features + variables_estructurales

resumen_caracteristicas = pd.DataFrame({
    "Tipo": df_viviendas[variables_revision_nulos].dtypes.astype(str),
    "Valores no nulos": df_viviendas[variables_revision_nulos].count(),
    "Valores nulos": df_viviendas[variables_revision_nulos].isna().sum(),
    "Porcentaje de nulos (%)": (
        df_viviendas[variables_revision_nulos].isna().mean() * 100
    ).round(2),
})

display(resumen_caracteristicas)

for variable in variables_revision_nulos:
    print(f"\n{variable}")
    print(df_viviendas[variable].value_counts(dropna=False).head(10))

### Interpretación

En las variables binarias, los valores ausentes no se reemplazan por `False` durante el preprocesamiento, ya que esa decisión haría indistinguibles “no informado” y “no dispone”. Los nulos permanecen en `viviendas_limpio.csv`. Si alguna técnica de modelización requiere valores completos, la imputación se aplicará después y se describirá de manera específica.


### 5.4 Variables restantes

Se examinan otras variables relevantes con valores ausentes, entre ellas `status`, `neighborhood`, `description`, `detailedType/subTypology` y `luxuryVisibility`.


In [ ]:
# =====================================================
# VARIABLES RESTANTES CON NULOS
# =====================================================

variables_restantes = [
    "detailedType/subTypology",
    "luxuryVisibility",
    "neighborhood",
    "status",
    "description",
]

resumen_restantes = pd.DataFrame({
    "Tipo": df_viviendas[variables_restantes].dtypes.astype(str),
    "Valores no nulos": df_viviendas[variables_restantes].count(),
    "Valores distintos": df_viviendas[variables_restantes].nunique(dropna=True),
    "Valores nulos": df_viviendas[variables_restantes].isna().sum(),
    "Porcentaje de nulos (%)": (
        df_viviendas[variables_restantes].isna().mean() * 100
    ).round(2),
})

display(resumen_restantes)

### Interpretación

Los valores ausentes de estas variables se explican, sobre todo, por información opcional o no facilitada por el anunciante. En esta fase se conservan sin imputación. En particular, `status` se mantendrá intacta y solo se transformará en el notebook de modelización.


## 6. Comprobación y eliminación de duplicados

Antes de exportar el dataset, se verifica si existen filas repetidas. Los duplicados pueden sobrerrepresentar ciertos anuncios y modificar las estadísticas descriptivas y los modelos.


In [ ]:
# =====================================================
# COMPROBACIÓN DE DUPLICADOS
# =====================================================

duplicados = df_viviendas.duplicated().sum()
print(f"Registros duplicados completos: {duplicados}")

duplicados_df = df_viviendas[
    df_viviendas.duplicated(keep=False)
]

if len(duplicados_df) > 0:
    columnas_revision = [
        c for c in ["url", "address", "price", "size", "ciudad"]
        if c in duplicados_df.columns
    ]
    display(duplicados_df[columnas_revision].head(20))

### Interpretación

La inspección indica que las filas repetidas pertenecen al mismo anuncio y reproducen exactamente la misma información. Por ello, los duplicados completos se eliminan antes de construir el conjunto procesado.


In [ ]:
# =====================================================
# ELIMINACIÓN Y VERIFICACIÓN DE DUPLICADOS
# =====================================================

registros_antes = len(df_viviendas)

df_viviendas = (
    df_viviendas
    .drop_duplicates()
    .reset_index(drop=True)
)

registros_despues = len(df_viviendas)

print(f"Registros antes: {registros_antes:,}")
print(f"Registros después: {registros_despues:,}")
print(f"Duplicados eliminados: {registros_antes - registros_despues:,}")
print(f"Duplicados restantes: {df_viviendas.duplicated().sum()}")

### Interpretación

Se eliminan **1.649 registros duplicados**, de modo que el conjunto pasa de **55.653 a 54.004 anuncios**. Después de esta operación no quedan duplicados completos.


## 7. Exportación del conjunto de datos procesado

El conjunto depurado se guarda como `viviendas_limpio.csv`. Este archivo conserva todas las tipologías inmobiliarias; el filtrado a pisos se realizará más adelante, cuando lo exija el objetivo analítico.


In [ ]:
# =====================================================
# EXPORTACIÓN DEL CONJUNTO PREPROCESADO
# =====================================================

ruta_salida = PROCESSED_DATA_DIR / "viviendas_limpio.csv"

df_viviendas.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8",
)

print(f"Archivo exportado: {ruta_salida}")
print(f"Dimensiones finales: {df_viviendas.shape[0]:,} registros x {df_viviendas.shape[1]} variables")

### Interpretación

El **conjunto procesado** contiene **54.004 registros y 62 variables**. Este archivo constituye la base común de los notebooks siguientes y conserva los nulos relevantes para que su tratamiento dependa de las necesidades concretas del análisis.


## 8. Resultados principales

- Conjunto integrado inicial: **55.653 registros y 535 variables**.
- Se eliminaron columnas completamente vacías, variables extremadamente escasas sin utilidad comparativa, recursos multimedia y metadatos administrativos no relevantes.
- El conjunto quedó reducido a **62 variables**.
- Los valores ausentes de las variables relevantes se conservaron cuando su significado no permitía tratarlos como ausencia de la característica.
- Se eliminaron **1.649 registros duplicados**.
- Conjunto procesado: **54.004 registros y 62 variables**.


## 9. Conclusiones de la fase

El preprocesamiento reduce de forma importante la complejidad del conjunto de datos sin eliminar las variables necesarias para el análisis inmobiliario.

La decisión metodológica principal consiste en **no transformar automáticamente los valores ausentes en ausencia de una característica**. Cuando un dato relevante no está informado, se mantiene como nulo en el conjunto procesado y su tratamiento se lleva a cabo más adelante solo si la técnica de análisis lo exige.

Tras la depuración estructural y la eliminación de **1.649 registros duplicados**, el archivo `viviendas_limpio.csv` contiene **54.004 anuncios y 62 variables**.

Este archivo representa el **conjunto de datos procesado de la fase de preparación**. Los notebooks posteriores partirán de este archivo y **definirán una muestra homogénea de pisos** para estudiar el comportamiento del precio y construir las regresiones por ciudad.


In [ ]:
# =====================================================
# EXPORTACIÓN DE TABLAS DE PREPROCESAMIENTO
# =====================================================

TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

tipos_datos.to_csv(
    TABLES_DIR / "02_tipos_datos.csv",
    index=False,
    encoding="utf-8-sig",
)

resumen_nulos.to_csv(
    TABLES_DIR / "02_resumen_nulos_por_intervalo.csv",
    index=False,
    encoding="utf-8-sig",
)

variables_vacias.reset_index().rename(
    columns={"index": "Variable"}
).to_csv(
    TABLES_DIR / "02_variables_completamente_vacias.csv",
    index=False,
    encoding="utf-8-sig",
)

categorias_vacias.to_csv(
    TABLES_DIR / "02_categorias_variables_vacias.csv",
    index=False,
    encoding="utf-8-sig",
)

variables_revision.reset_index().rename(
    columns={"index": "Variable"}
).to_csv(
    TABLES_DIR / "02_variables_revision_mas_90_nulos.csv",
    index=False,
    encoding="utf-8-sig",
)

resumen_price_drop.reset_index().rename(
    columns={"index": "Variable"}
).to_csv(
    TABLES_DIR / "02_resumen_reducciones_precio.csv",
    index=False,
    encoding="utf-8-sig",
)

resumen_parking.reset_index().rename(
    columns={"index": "Variable"}
).to_csv(
    TABLES_DIR / "02_resumen_aparcamiento.csv",
    index=False,
    encoding="utf-8-sig",
)

resumen_caracteristicas.reset_index().rename(
    columns={"index": "Variable"}
).to_csv(
    TABLES_DIR / "02_resumen_caracteristicas.csv",
    index=False,
    encoding="utf-8-sig",
)

resumen_restantes.reset_index().rename(
    columns={"index": "Variable"}
).to_csv(
    TABLES_DIR / "02_resumen_variables_restantes.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Tablas del Notebook 02 exportadas en:", TABLES_DIR)
